In [1]:
import pandas as pd
import warnings
import sys
from pathlib import Path
from datasets import load_dataset
import re
from collections import Counter

In [2]:
warnings.simplefilter(action='ignore', category=FutureWarning) 

In [3]:
# Assuming the working directory is root/part1 or root/part2
root = Path.cwd().parent
sys.path.insert(0, str(root))
import part1

In [4]:
# save models and outputs there to keep the directory tree organised
results_folder = Path("./results")
results_folder.mkdir(parents=True, exist_ok=True) 
results_folder

WindowsPath('results')

In [5]:
# Login using e.g. `huggingface-cli login` to access this dataset
df = load_dataset("mteb/imdb")


value = df['train'][0]
print(value)

review = value['text']
print(review)

binary = value['label']
print(binary)

target = value['label_text']
print(target)


{'text': 'I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ever tried to enter this country, therefore being a fan of films considered "controversial" I really had to see this for myself.<br /><br />The plot is centered around a young Swedish drama student named Lena who wants to learn everything she can about life. In particular she wants to focus her attentions to making some sort of documentary on what the average Swede thought about certain political issues such as the Vietnam War and race issues in the United States. In between asking politicians and ordinary denizens of Stockholm about their opinions on politics, she has sex with her drama teacher, classmates, and married men.<br /><br />What kills me about I AM CURIOUS-YELLOW is that 40 years ago, this was considered pornographic. Really, the sex and nudity scenes are few and far be

In [6]:
imdb_outcome_space = part1.clslib.NamedClasses(['negative', 'positive'])
print(imdb_outcome_space)

0: negative, 1: positive


In [7]:
df_train = pd.DataFrame(df['train'])
display(df_train)

df_test = pd.DataFrame(df['test'])
display(df_test)

,text,label,label_text
0,I rented I AM CURIOUS-YELLOW from my video sto...,0,negative
1,"""I Am Curious: Yellow"" is a risible and preten...",0,negative
2,If only to avoid making this type of film in t...,0,negative
3,This film was probably inspired by Godard's Ma...,0,negative
4,"Oh, brother...after hearing about this ridicul...",0,negative
...,...,...,...
24899,A hit at the time but now better categorised a...,1,positive
24900,I love this movie like no other. Another time ...,1,positive
24901,This film and it's sequel Barry Mckenzie holds...,1,positive
24902,'The Adventures Of Barry McKenzie' started lif...,1,positive


,text,label,label_text
0,I love sci-fi and am willing to put up with a ...,0,negative
1,"Worth the entertainment value of a rental, esp...",0,negative
2,its a totally average film with a few semi-alr...,0,negative
3,STAR RATING: ***** Saturday Night **** Friday ...,0,negative
4,"First off let me say, If you haven't enjoyed a...",0,negative
...,...,...,...
24673,Just got around to seeing Monster Man yesterda...,1,positive
24674,I got this as part of a competition prize. I w...,1,positive
24675,I got Monster Man in a box set of three films ...,1,positive
24676,"Five minutes in, i started to feel how naff th...",1,positive


In [8]:
def clean(text):
    no_html = re.sub(r"</?(br|i|em|hr)\s*/?>", " ", text)
    no_space = " ".join(no_html.split())
    return no_space

In [9]:
dftrain_html = []

for i in df['train']['text']:
    dftrain_html.append(clean(i))

In [10]:
dftest_html = []

for i in df['test']['text']:
    dftest_html.append(clean(i))

In [11]:
print(sum("<br" in t for t in dftrain_html))   # should be 0
print(sum(">" in t for t in dftrain_html))     # should be small, not ~14,600
print(dftrain_html[0][:500])                   # read it: no tags, no stray >

0
42
I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ever tried to enter this country, therefore being a fan of films considered "controversial" I really had to see this for myself. The plot is centered around a young Swedish drama student named Lena who wants to learn everything she can about life. In particular she wants to focus her attentions to mak


In [12]:
train_labels = list(df['train']['label'])
test_labels  = list(df['test']['label'])

print(sum("\t" in t for t in dftrain_html))
print(sum("\n" in t for t in dftrain_html))

0
0


We searched for anything that looks like a tag and inspected everything we found. Only real HTML (<br />, <i>, <em>, <hr>) was removed, replaced by a space so words don’t stick together. Things like <sigh> were kept because the reviewer wrote them. We also turned tabs and repeated spaces into single spaces, because the tokenizer only splits on normal spaces.

In [13]:
train_total = len(train_labels)
train_neg = Counter(train_labels)[0] / train_total * 100
train_pos = Counter(train_labels)[1] / train_total * 100

print(Counter(train_labels))  
print(train_neg)
print(train_pos)

Counter({1: 12472, 0: 12432})
49.91969161580469
50.080308384195305


In [14]:
test_total = len(test_labels)
test_neg = Counter(test_labels)[0] / test_total * 100
test_pos = Counter(test_labels)[1] / test_total * 100

print(Counter(test_labels))
print(test_neg)
print(test_pos)

Counter({1: 12412, 0: 12266})
49.70418996677202
50.29581003322797


In [15]:
df_train["text"] = df_train["text"].apply(clean)
df_test["text"]  = df_test["text"].apply(clean)

In [16]:
print(df_train["text"].duplicated().sum())               
print(df_test["text"].isin(df_train["text"]).sum())      
df_train = df_train.drop_duplicates(subset="text").reset_index(drop=True)
print(len(df_train))                                     

2
0
24902


In [17]:
dftrain_html = df_train["text"].tolist()
train_labels = df_train["label"].tolist()
dftest_html  = df_test["text"].tolist()
test_labels  = df_test["label"].tolist()

In [18]:
vocab_size = 1000
min_frequency = 2
tokenizer = part1.experiment.train_tokenizer(dftrain_html, vocab_size, min_frequency)

In [19]:
tokenizer.save(f"{results_folder}/BPE_tokenizer_vs_{vocab_size}_mf_{min_frequency}.json")

In [20]:
print(tokenizer.get_vocab_size())    # 1000
print(tokenizer.encode("This movie was not good, the acting was terrible.").tokens)

1000
['▁This', '▁movie', '▁was', '▁not', '▁good', ',', '▁the', '▁acting', '▁was', '▁ter', 'ri', 'ble', '.']


In [21]:
%%time
my_seeds = [0] # [0, 3, 5] # add more seeds here for multiple runs of cross validation, eg. [0, 3, 5]  
num_folds = 5  # more makes each cross validation run more statistically informative, but it can get slow
my_cross_validation_sets = []

for seed in my_seeds:  
    my_cross_validation_sets.append(
        part1.experiment.cross_validation_split(
            # we pair the texts with their labels
            pairs=list(zip(dftrain_html, train_labels)), 
            k=num_folds,
            seed=seed
        )
    )

CPU times: total: 46.9 ms
Wall time: 103 ms
